# TRON vGPU Diagnostic Notebook
Inspect scheduler, gateway, worker, and database state.

In [ ]:
import os
import json
import sqlite3
from pathlib import Path
import requests
import socket

root = Path('.')
print('cwd', root.resolve())
for port in [9002, 9003]:
    s = socket.socket()
    s.settimeout(1)
    try:
        s.connect(('127.0.0.1', port))
        print(f'port {port} open')
    except Exception as exc:
        print(f'port {port} closed or unreachable: {exc}')
    finally:
        s.close()

urls = [
    'http://127.0.0.1:9002/health',
    'http://127.0.0.1:9003/health',
    'http://127.0.0.1:9002/workers',
    'http://127.0.0.1:9002/jobs',
    'http://127.0.0.1:9002/ledger',
    'http://127.0.0.1:9002/next_job',
    'http://127.0.0.1:9002/submit_job',
]
for url in urls:
    try:
        if url.endswith('/submit_job'):
            r = requests.post(url, json={
                'task_type': 'vgpu_task',
                'payload': {'task_kind': 'openai_chat', 'prompt': 'diagnostic test', 'duration': 0.1},
                'priority': 1,
                'requires_gpu': True,
                'required_vram_gb': 0.5,
                'required_cuda_cores': 64,
            }, timeout=10)
        else:
            r = requests.get(url, timeout=10)
        print(url, r.status_code, r.text[:400])
    except Exception as exc:
        print(url, 'error', type(exc).__name__, exc)

db_path = Path('tron_vgpu_master.db')
print('db exists', db_path.exists(), db_path.stat().st_size if db_path.exists() else None)
if db_path.exists():
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    for table in ['workers', 'jobs']:
        try:
            cur.execute(f'SELECT COUNT(*) FROM {table}')
            print(table, 'count', cur.fetchone()[0])
            cur.execute(f'SELECT * FROM {table} ORDER BY rowid DESC LIMIT 5')
            rows = cur.fetchall()
            print(table, 'sample rows:', rows)
        except Exception as exc:
            print('failed reading', table, exc)
    conn.close()